In [22]:
import os
from pathlib import Path

from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [23]:
DOCUMENTS_FOLDER = "Docs"
INDEX_PATH = "faiss_index"
AWS_REGION = "us-east-1"
AWS_EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL = "meta.llama3-70b-instruct-v1:0"

In [24]:
def load_documents(folder_path):
    all_documents = []

    for file_path in Path(folder_path).rglob("*"):
        try:
            if file_path.suffix.lower() == '.pdf':
                print(f"Loading PDF: {file_path}")
                
                loader = PyPDFLoader(str(file_path))

                docs = loader.load()

                for doc in docs:
                    doc.metadata["source"] = str(file_path)
                    doc.metadata["file_name"] = file_path.name 
                    doc.metadata["document_type"] = "pdf"
                all_documents.extend(docs)
                
            elif file_path.suffix.lower() == '.txt':
                print(f"Loading Text Documents: {file_path}")

                loader = TextLoader(
                    str(file_path),
                    encoding = "utf-8"
                )

                docs = loader.load()

                for doc in docs:
                    doc.metadata["source"] = str(file_path)
                    doc.metadata["file_name"] = file_path.name
                    doc.metadata["document_type"] = "txt"
                all_documents.extend(docs)

        except Exception as e:
            print(f"Error Loading: {file_path}")
            print(e)

    return all_documents

In [25]:
documents = load_documents(DOCUMENTS_FOLDER)

In [26]:


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 100,
    separators=[
        "\n\n",
        "\n",
        " ",
        ""
    ]
)

In [ ]:
chunks = text_splitter.split_documents(documents)
print(f"Total Chunks Created: {len(chunks)}")

embeddings = BedrockEmbeddings(
    model_id = AWS_EMBEDDING_MODEL,
    region_name = AWS_REGION
)

Total Chunks Created: 0


In [30]:
embeddings

BedrockEmbeddings(client=<botocore.client.BedrockRuntime object at 0x7952d436dc40>, region_name='us-east-1', credentials_profile_name=None, model_id='amazon.titan-embed-text-v2:0', model_kwargs=None, endpoint_url=None, normalize=False, config=None)

In [32]:
if os.path.exists(INDEX_PATH):
    print("\n Loading Existing FAISS Index")

    vectorstore = FAISS.load_local(
        INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization = True
    )

else:
    print("\n Creating New Vectore Store DB")

    vector_store = FAISS.from_documents(
        chunks,
        embeddings
    )

    vector_store.save_local(INDEX_PATH)

print("\n Vector Store is Ready")


 Creating New Vectore Store DB


IndexError: list index out of range